In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q15_rewrite_small/checkpoints/pre_cell_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 2 ###

# Optimized GPU workflow using cudf.pandas
# Define date bounds as strings to push comparisons to the GPU
date_start = "1996-01-01"
date_end = "1996-04-01"

# Filter, compute revenue parts, aggregate and pick top supplier in one pipeline
top_revenue = (
    lineitem
    .loc[
        (lineitem.L_SHIPDATE >= date_start) & (lineitem.L_SHIPDATE < date_end),
        ["L_SUPPKEY", "L_EXTENDEDPRICE", "L_DISCOUNT"]
    ]
    .assign(REVENUE_PARTS=lambda df: df.L_EXTENDEDPRICE * (1 - df.L_DISCOUNT))
    .groupby("L_SUPPKEY")["REVENUE_PARTS"].sum()
    .reset_index()
    .nlargest(1, "REVENUE_PARTS")
    .rename(columns={"L_SUPPKEY": "S_SUPPKEY", "REVENUE_PARTS": "TOTAL_REVENUE"})
)

# Join with supplier details and select final columns
total = (
    top_revenue
    .merge(
        supplier[["S_SUPPKEY", "S_NAME", "S_ADDRESS", "S_PHONE"]],
        on="S_SUPPKEY"
    )
    [["S_SUPPKEY", "S_NAME", "S_ADDRESS", "S_PHONE", "TOTAL_REVENUE"]]
)
